## Data Loading

In [88]:
import sqlite3
import pandas as pd
import yaml
import warnings
warnings.filterwarnings('ignore')

# 1. Load Categories from YAML
def load_categories(yml_path):
    with open(yml_path, 'r') as file:
        data = yaml.safe_load(file)
    
    category_map = {}
    for category, items in data.items():
        if isinstance(items, dict):
            for item_key in items.keys():
                category_map[item_key.upper()] = category
    return category_map

category_map = load_categories('price.yml')

# 2. Load SQLite Database
conn = sqlite3.connect('upgradepvp.db')
df_purchases = pd.read_sql_query("SELECT * FROM purchase", conn)
df_deaths = pd.read_sql_query("SELECT * FROM death", conn)
conn.close()

# Ensure datetime format for accurate chronological sorting
df_purchases['created_at'] = pd.to_datetime(df_purchases['created_at'])
df_deaths['created_at'] = pd.to_datetime(df_deaths['created_at'])

print("Adatok sikeresen betöltve!")

Adatok sikeresen betöltve!


## 1D Data Preprocessing (Item -> Item)

In [89]:
import pandas as pd
import re
from mlxtend.preprocessing import TransactionEncoder

# Helper to clean items AND replace books with their enchants
def parse_item(raw_item):
    if not isinstance(raw_item, str):
        return str(raw_item), None
    parts = raw_item.split('$')
    base_item = parts[0]
    detail = None
    if len(parts) > 1 and parts[1] != 'null':
        detail = re.sub(r'§[0-9a-fk-or]', '', parts[1])
        
    # --- Új logika: Könyv cseréje a varázslatra ---
    if base_item == 'ENCHANTED_BOOK' and detail:
        base_item = detail  # A tárgy neve ezentúl a varázslat lesz (pl. "Power")
        detail = None       # A részletet töröljük, mert már beépült a tárgy nevébe

    return base_item, detail

events_p = df_purchases[['game_id', 'player_uuid', 'created_at', 'item']].copy()
events_p['event_type'] = 'purchase'

events_d = df_deaths[['game_id', 'dead_uuid', 'created_at']].copy()
events_d.rename(columns={'dead_uuid': 'player_uuid'}, inplace=True)
events_d['event_type'] = 'death'
events_d['item'] = None

events = pd.concat([events_p, events_d]).sort_values(by=['game_id', 'created_at'])

dataset_1d = []
player_states = {}

for _, row in events.iterrows():
    key = (row['game_id'], row['player_uuid'])
    
    if key not in player_states:
        player_states[key] = {'inventory': set(), 'keep_inv': False}
        
    state = player_states[key]
    
    if row['event_type'] == 'purchase':
        # Használjuk az új parse_item függvényt
        base_item, _ = parse_item(row['item'])
        
        if 'KEEPINVENTORY' in base_item.upper():
            state['keep_inv'] = True
            
        state['inventory'].add(base_item)
        
        if len(state['inventory']) > 0:
            dataset_1d.append(list(state['inventory']))
        
    elif row['event_type'] == 'death':
        if not state['keep_inv']:
            state['inventory'].clear()

te_1d = TransactionEncoder()
te_ary_1d = te_1d.fit(dataset_1d).transform(dataset_1d)
df_trans_1d = pd.DataFrame(te_ary_1d, columns=te_1d.columns_)

print(f"1D Tranzakciók száma (csak tárgyak/varázslatok): {len(dataset_1d)}")

1D Tranzakciók száma (csak tárgyak/varázslatok): 688


## 1D Apriori & FP-Growth

In [90]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Segédfüggvény a frozenset formázásához
def format_itemset(itemset):
    return ", ".join(list(itemset))

print("--- 1D FP-GROWTH: Gyakori Elemhalmazok ---")
freq_items_1d = fpgrowth(df_trans_1d, min_support=0.05, use_colnames=True)
freq_items_1d['Tárgyak'] = freq_items_1d['itemsets'].apply(format_itemset)
display(freq_items_1d[['Tárgyak', 'support']].sort_values(by='support', ascending=False).head(5))

print("\n--- 1D ASSZOCIÁCIÓS SZABÁLYOK (Tárgy -> Tárgy) ---")
rules_1d = association_rules(freq_items_1d, metric="confidence", min_threshold=0.3)

# Oszlopok tisztítása és magyarítása
rules_1d['Törzs (A)'] = rules_1d['antecedents'].apply(format_itemset)
rules_1d['Fej (B)'] = rules_1d['consequents'].apply(format_itemset)

output_1d = rules_1d[['Törzs (A)', 'Fej (B)', 'support', 'confidence', 'lift']]
output_1d.columns = ['Ha megveszi ezt (A)', 'Akkor megveszi ezt is (B)', 'Támasz', 'Konfidencia', 'Lift']

display(output_1d.sort_values(by='Lift', ascending=False).head(10))

--- 1D FP-GROWTH: Gyakori Elemhalmazok ---


,Tárgyak,support
6,SNOW_BALL,0.598837
9,IRON_SWORD,0.321221
2,GOLD_SWORD,0.284884
28,"SNOW_BALL, IRON_SWORD",0.254360
10,Fire Aspect,0.191860



--- 1D ASSZOCIÁCIÓS SZABÁLYOK (Tárgy -> Tárgy) ---


,Ha megveszi ezt (A),Akkor megveszi ezt is (B),Támasz,Konfidencia,Lift
3529,"SNOW_BALL, IRON_LEGGINGS","IRON_SWORD, IRON_HELMET, Fire Aspect, IRON_BOOTS",0.05814,0.769231,12.908068
3527,"SNOW_BALL, IRON_BOOTS","IRON_SWORD, IRON_LEGGINGS, Fire Aspect, IRON_HELMET",0.05814,0.769231,12.908068
1366,"IRON_SWORD, Fire Aspect, IRON_HELMET","SNOW_BALL, IRON_LEGGINGS",0.05814,0.975610,12.908068
4788,"Knockback, IRON_SWORD, Fire Aspect, IRON_HELMET","SNOW_BALL, IRON_LEGGINGS, IRON_CHESTPLATE, IRON_BOOTS",0.05814,0.975610,12.908068
4785,"IRON_CHESTPLATE, IRON_SWORD, Fire Aspect, IRON_HELMET","Knockback, SNOW_BALL, IRON_LEGGINGS, IRON_BOOTS",0.05814,0.975610,12.908068
4837,"IRON_SWORD, Fire Aspect, IRON_HELMET","SNOW_BALL, IRON_BOOTS, IRON_CHESTPLATE, Knockback, IRON_LEGGINGS",0.05814,0.975610,12.908068
4087,"Knockback, IRON_SWORD, Fire Aspect, IRON_HELMET","SNOW_BALL, IRON_CHESTPLATE, IRON_BOOTS",0.05814,0.975610,12.908068
4712,"SNOW_BALL, IRON_BOOTS, IRON_CHESTPLATE, Knockback, IRON_LEGGINGS","IRON_SWORD, Fire Aspect, IRON_HELMET",0.05814,0.769231,12.908068
4719,"Fire Aspect, IRON_HELMET, IRON_BOOTS, IRON_CHESTPLATE, IRON_SWORD","Knockback, SNOW_BALL, IRON_LEGGINGS",0.05814,0.975610,12.908068
4789,"IRON_SWORD, IRON_LEGGINGS, Fire Aspect, IRON_HELMET","Knockback, SNOW_BALL, IRON_CHESTPLATE, IRON_BOOTS",0.05814,0.975610,12.908068


## Multi-Dimensional Data Preprocessing

In [ ]:
import pandas as pd
import re
import yaml
from mlxtend.preprocessing import TransactionEncoder

# 1. Töltsük be a YAML fájlt
with open('price.yml', 'r') as file:
    yaml_data = yaml.safe_load(file)

# 2. Dinamikus kategória-térkép építése a YAML alapján (Hardkódolás nélkül)
def build_category_map(yaml_data):
    cat_map = {}
    for top_cat, content in yaml_data.items():
        # A varázslatokat és a győzelmi pontot kihagyjuk a kategóriákból
        if top_cat in ['Win', 'Enchants'] or not isinstance(content, dict):
            continue
            
        for sub_cat, value in content.items():
            if isinstance(value, dict):
                # Többszintű kategória (pl. Armor -> Leather -> Helmet)
                for piece in value.keys():
                    mat = sub_cat.upper()
                    category_name = f"{top_cat}_{sub_cat}" # pl. Armor_Leather
                    
                    # Alap leképezés (pl. LEATHERHELMET) és Minecraft eltérések lefedése
                    cat_map[f"{mat}{piece.upper()}"] = category_name
                    if mat == "CHAIN": cat_map[f"CHAINMAIL{piece.upper()}"] = category_name
                    if mat == "GOLD": cat_map[f"GOLDEN{piece.upper()}"] = category_name
            else:
                # Egyszintű kategória (pl. Swords -> Wooden, Other -> Bow)
                if top_cat == 'Swords':
                    mat = sub_cat.upper()
                    cat_map[f"{mat}SWORD"] = top_cat
                    if mat == "WOODEN": cat_map["WOODSWORD"] = top_cat
                    if mat == "GOLD": cat_map["GOLDSWORD"] = top_cat
                else:
                    # Egyéb tárgyak (pl. Bow, SnowBall -> SNOWBALL)
                    cat_map[sub_cat.upper()] = top_cat
                    
    return cat_map

# Felépítjük a szótárat a feldolgozás előtt
category_dict = build_category_map(yaml_data)

def get_category(base_item, cat_map):
    # Eltávolítjuk az alulvonásokat (pl. SNOW_BALL -> SNOWBALL, DIAMOND_SWORD -> DIAMONDSWORD)
    # Így könnyen illeszkedik a YAML-ből generált kulcsokra.
    clean_item = base_item.upper().replace('_', '')
    return cat_map.get(clean_item, None)

# 3. Tárgyak és könyvek tisztítása
def parse_multidim_item(raw_item):
    if not isinstance(raw_item, str):
        return str(raw_item), None
    parts = raw_item.split('$')
    base_item = parts[0]
    detail = None
    if len(parts) > 1 and parts[1] != 'null':
        detail = re.sub(r'§[0-9a-fk-or]', '', parts[1])
        
    if base_item == 'ENCHANTED_BOOK' and detail:
        base_item = detail  
        detail = None       

    return base_item, detail

dataset_md = []
player_states_md = {}

for _, row in events.iterrows():
    key = (row['game_id'], row['player_uuid'])
    if key not in player_states_md:
        player_states_md[key] = {'inventory': set(), 'keep_inv': False}
        
    state = player_states_md[key]
    
    if row['event_type'] == 'purchase':
        base_item, detail = parse_multidim_item(row['item'])
        
        if 'KEEPINVENTORY' in base_item.upper():
            state['keep_inv'] = True
            
        state['inventory'].add(f"Tárgy: {base_item}")
        
        if detail:
            state['inventory'].add(f"Varázslat: {detail}")
            
        # Dinamikus kategória lekérdezés a YAML szótárból
        cat = get_category(base_item, category_dict)
        if cat:
            state['inventory'].add(f"Kategória: {cat}")
        
        current_tx = list(state['inventory'])
        if base_item == 'DIAMOND':
            current_tx.append("Kimenetel: GYŐZELEM")
            
        dataset_md.append(current_tx)
        
    elif row['event_type'] == 'death':
        if len(state['inventory']) > 0:
            current_tx = list(state['inventory']) + ["Kimenetel: HALÁL"]
            dataset_md.append(current_tx)
            
        if not state['keep_inv']:
            state['inventory'].clear()

te_md = TransactionEncoder()
te_ary_md = te_md.fit(dataset_md).transform(dataset_md)
df_trans_md = pd.DataFrame(te_ary_md, columns=te_md.columns_)

print(f"Többdimenziós Tranzakciók száma: {len(dataset_md)}")

Többdimenziós Tranzakciók száma: 777


## Multi-Dimensional Rules & Presentation Metrics

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Segédfüggvény a frozenset formázásához
def format_itemset(itemset):
    return ", ".join(list(itemset))

# Gyakori elemhalmazok
freq_items_md = fpgrowth(df_trans_md, min_support=0.1, max_len=3, use_colnames=True)
rules_md = association_rules(freq_items_md, metric="confidence", min_threshold=0.3)

# Mutatószámok (Metrics) kiszámítása a dia alapján
def calculate_covariance(row):
    p_A = row['antecedent support']
    p_B = row['consequent support']
    p_AB = row['support']
    return p_AB - (p_A * p_B)

def calculate_correlation(row):
    cov = row['covariance']
    p_A = row['antecedent support']
    p_B = row['consequent support']
    denominator = np.sqrt(p_A * p_B * (1 - p_A) * (1 - p_B))
    return cov / denominator if denominator != 0 else 0

rules_md['covariance'] = rules_md.apply(calculate_covariance, axis=1)
rules_md['correlation'] = rules_md.apply(calculate_correlation, axis=1)

# Oszlopok emberi olvashatóságának javítása
rules_md['Törzs (A)'] = rules_md['antecedents'].apply(format_itemset)
rules_md['Fej (B)'] = rules_md['consequents'].apply(format_itemset)

mutatoszamok = rules_md[['Törzs (A)', 'Fej (B)', 'support', 'confidence', 'lift', 'covariance', 'correlation']].copy()
mutatoszamok.columns = ['Feltétel (A)', 'Következmény (B)', 'Támasz (Support)', 'Konfidencia', 'Lift', 'Kovariancia', 'Korreláció']

# Érdekes szabályok szűrése (pl. kategóriák, varázslatok, győzelem közötti kapcsolatok)
filtered_rules = mutatoszamok[
    (mutatoszamok['Feltétel (A)'].str.contains("Kategória|Varázslat|Kimenetel")) |
    (mutatoszamok['Következmény (B)'].str.contains("Kategória|Varázslat|Kimenetel"))
]

print("--- Többdimenziós Szabályok Mutatószámai (Prezentáció Alapján) ---")
display(filtered_rules.sort_values(by='Lift', ascending=False).head(15))

--- Többdimenziós Szabályok Mutatószámai (Prezentáció Alapján) ---


,Feltétel (A),Következmény (B),Támasz (Support),Konfidencia,Lift,Kovariancia,Korreláció
380,"Tárgy: IRON_LEGGINGS, Kategória: Swords",Tárgy: IRON_HELMET,0.104247,0.941860,8.509600,0.091997,0.934625
383,Tárgy: IRON_HELMET,"Tárgy: IRON_LEGGINGS, Kategória: Swords",0.104247,0.941860,8.509600,0.091997,0.934625
25,Tárgy: BOW,"Tárgy: ARROW, Kategória: Other",0.100386,0.962963,8.406991,0.088445,0.908826
23,"Tárgy: BOW, Kategória: Other",Tárgy: ARROW,0.100386,0.962963,8.406991,0.088445,0.908826
21,"Tárgy: ARROW, Kategória: Other",Tárgy: BOW,0.100386,0.876404,8.406991,0.088445,0.908826
24,Tárgy: ARROW,"Tárgy: BOW, Kategória: Other",0.100386,0.876404,8.406991,0.088445,0.908826
405,Tárgy: IRON_LEGGINGS,"Kategória: Armor_Iron, Tárgy: IRON_BOOTS",0.100386,0.857143,8.222222,0.088177,0.897361
403,"Tárgy: IRON_LEGGINGS, Kategória: Armor_Iron",Tárgy: IRON_BOOTS,0.100386,0.857143,8.222222,0.088177,0.897361
404,"Kategória: Armor_Iron, Tárgy: IRON_BOOTS",Tárgy: IRON_LEGGINGS,0.100386,0.962963,8.222222,0.088177,0.897361
406,Tárgy: IRON_BOOTS,"Tárgy: IRON_LEGGINGS, Kategória: Armor_Iron",0.100386,0.962963,8.222222,0.088177,0.897361
